In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import argparse
import os

from utils.columns import *
from utils.imputation import impute_icustay_ids, fill_outliers, fill_stepwise, sample_and_hold, fixgaps, knn_impute
from utils.derived_features import calculate_onset, compute_pao2_fio2, compute_shock_index, compute_sofa, compute_sirs

In [ ]:
in_dir = os.path.join("./processed_files")
out_dir = os.path.join("./intermediate_files")

### 1. REFTABLE 처리

In [ ]:
paths = [p for p in os.listdir(in_dir) if p.startswith("labs") or p.startswith("ce")]

In [ ]:
for file_name in tqdm(paths):
    if file_name.startswith("labs"):
        ref_vals = REF_LABS
    elif file_name.startswith("ce"):
        ref_vals = REF_VITALS

    df = pd.read_csv(os.path.join(in_dir, file_name), dtype={C_ITEMID: int})

    # Build a converter that maps itemid codes to the row of the reference list in which they are present
    converter = {x: i + 1 for i, ids in enumerate(ref_vals) for x in ids}

    # Replace each itemid using the converter
    df[C_ITEMID].replace(converter, inplace=True)
    df.to_csv(os.path.join(out_dir, file_name), index=False)

### 2. generate bacterio
Produce bacterio dataframe by combining microbio and culture

In [ ]:
demog = pd.read_csv(os.path.join(in_dir, 'demog.csv'))
abx = pd.read_csv(os.path.join(in_dir, 'abx.csv'))

In [ ]:
culture = pd.read_csv(os.path.join(in_dir, 'culture.csv'))
microbio = pd.read_csv(os.path.join(in_dir, 'microbio.csv'))

In [ ]:
# use chartdate to fill in empty charttimes
ii = microbio[C_CHARTTIME].isnull()
microbio.loc[ii, C_CHARTTIME] = microbio.loc[ii, C_CHARTDATE]
microbio.loc[:, C_CHARTDATE] = 0

In [ ]:
cols = [C_SUBJECT_ID, C_HADM_ID, C_ICUSTAYID, C_CHARTTIME]
bacterio = pd.concat([microbio[cols], culture[cols]], ignore_index=True)

In [ ]:
tqdm.pandas()

In [ ]:
# Impute icu stay IDs from demog
print("Bacterio has {}/{} null ICU stay IDs, imputing...".format(pd.isna(bacterio[C_ICUSTAYID]).sum(), len(bacterio)))
bacterio.loc[pd.isna(bacterio[C_ICUSTAYID]), C_ICUSTAYID] = impute_icustay_ids(demog, bacterio[pd.isna(bacterio[C_ICUSTAYID])])
print("Now:", pd.isna(bacterio[C_ICUSTAYID]).sum(), "nulls")

In [ ]:
bacterio = bacterio[~pd.isna(bacterio[C_ICUSTAYID])]
bacterio.to_csv(os.path.join(out_dir, "bacterio.csv"), index=False)

### 2.5 impute icustay ids on clinical note

In [ ]:
note = pd.read_csv(os.path.join(in_dir, 'note.csv'))

In [ ]:
note['text'] = '['+note['category'].astype(str) + ']' + note['text']
note.drop(['category'], axis=1, inplace=True)

In [ ]:
print("clinical note has {}/{} null ICU stay IDs, imputing...".format(pd.isna(note[C_ICUSTAYID]).sum(), len(note)))
note.loc[pd.isna(note[C_ICUSTAYID]), C_ICUSTAYID] = impute_icustay_ids(demog, note[pd.isna(note[C_ICUSTAYID])])
print("Now:", pd.isna(note[C_ICUSTAYID]).sum(), "nulls")

In [ ]:
note = note[~pd.isna(note[C_ICUSTAYID])]

In [ ]:
note

In [ ]:
note.to_csv(os.path.join(out_dir, "note.csv"), index=False)

### 3. Process abx

In [ ]:
 # Remove abx entries with no start date or ICU stay ID
print("Trimming unusable entries [abx]")
abx = abx[(~pd.isna(abx[C_STARTDATE])) & (~pd.isna(abx[C_ICUSTAYID]))]

In [ ]:
print("Abx has {}/{} nulls, imputing...".format(pd.isna(abx[C_ICUSTAYID]).sum(), len(abx)))
abx.loc[pd.isna(abx[C_ICUSTAYID]), C_ICUSTAYID] = impute_icustay_ids(demog, abx[pd.isna(abx[C_ICUSTAYID])])
print("Now:", pd.isna(abx[C_ICUSTAYID]).sum(), "nulls")
abx.to_csv(os.path.join(out_dir, "abx.csv"), index=False)

### 4. process demog

In [ ]:
# Correct nans in demog
print("Correcting nans [demog]")
demog.loc[pd.isna(demog[C_MORTA_90]), C_MORTA_90] = 0
demog.loc[pd.isna(demog[C_MORTA_HOSP]), C_MORTA_HOSP] = 0
demog.loc[pd.isna(demog[C_ELIXHAUSER]), C_ELIXHAUSER] = 0
demog.to_csv(os.path.join(out_dir, "demog.csv"), index=False)

### 5. process inputMV

In [ ]:
# compute normalized rate of infusion and add it as a column to inputMV
# if we give 100 ml of hypertonic fluid (600 mosm/l) at 100 ml/h (given in 1h) it is 200 ml of NS equivalent
# so the normalized rate of infusion is 200 ml/h (different volume in same duration)
print("Computing normalized rate of infusion [fluid_mv]")
inputMV = pd.read_csv(os.path.join(in_dir, 'fluid_align_with_snuh_mv.csv'))
inputMV.loc[:, C_NORM_INFUSION_RATE] = inputMV[:][C_TEV] * inputMV[:][C_RATE] / inputMV[:][C_AMOUNT]
inputMV.to_csv(os.path.join(out_dir, "fluid_align_with_snuh_mv.csv"), index=False)

### 6. process mechvent

In [ ]:
print("Correcting nans [mechvent]")
mechvent = pd.read_csv(os.path.join(in_dir, 'mechvent.csv'))
mechvent = mechvent[~pd.isna(mechvent[C_ICUSTAYID])]
mechvent.to_csv(os.path.join(out_dir, "mechvent.csv"), index=False)

### 7. process vaso_mv

In [ ]:
print("Correcting nans [vaso_mv]")
vaso_mv = pd.read_csv(os.path.join(in_dir, 'vaso_align_with_snuh_mv.csv'))
vaso_mv = vaso_mv[~pd.isna(vaso_mv[C_ICUSTAYID])]
vaso_mv.to_csv(os.path.join(out_dir, "vaso_align_with_snuh_mv.csv"), index=False)

### 8. calculate sepsis onset
1. 항생제 투여 후 24시간 이내에 bacterio 이벤트 발생의 경우 onset time을 항생제 투여 시점으로 결정
2. 항생제 투여 시점이 bacterio 이벤트 시점보다 늦거나 같으며, 시간 차이가 72시간 이내인 경우 onset time을 bacterio이벤트 시점으로 설정

In [ ]:
abx = pd.read_csv(os.path.join(out_dir, "abx.csv"))
bacterio = pd.read_csv(os.path.join(out_dir, "bacterio.csv"))

In [ ]:
onset_data = pd.DataFrame([onset for onset in (calculate_onset(abx, bacterio, stay_id) for stay_id in tqdm(abx[C_ICUSTAYID].unique())) if onset is not None])

In [ ]:
onset_data.to_csv(os.path.join(out_dir, "sepsis_onset.csv"), index=False)

### 9. build patient states

In [ ]:
class ChartEvents:
    """
    An object that manages a set of chart event tables, and 
    supports retrieving all chart events for a given ICU stay ID.
    """
    def __init__(self, ce_dfs, stay_id_col=C_ICUSTAYID):
        super().__init__()
        self.dfs = ce_dfs
        self.ranges = [(df[stay_id_col].min(), df[stay_id_col].max())
                       for df in ce_dfs]
        self.stay_id_col = stay_id_col
        
    def fetch(self, stay_id):
        results = []
        for df, (min_id, max_id) in zip(self.dfs, self.ranges):
            if stay_id >= min_id and stay_id <= max_id:
                results.append(df[df[self.stay_id_col] == stay_id])
        if results:
            return pd.concat(results)
        return None

def time_window(df, col, center_time, lower_window, upper_window):
    """
    Returns all rows in the given data frame whose timestamp in column 'col' fall
    between center_time - lower_window and center_time + upper_window.
    """
    return df[(df[col] >= center_time - lower_window) & (df[col] <= center_time + upper_window)]

def build_patient_states(chart_events, onset_data, demog, labU, MV, MV_procedure, winb4, winaft):
    """
    Builds the patient states dataframe without imputation. Returns two dataframes,
    one containing the patient states and the other containing the qstime
    dataframe, which maps ICU stay IDs to relevant timestamps in the stay.
    """    
    combined_data = []
    infection_times = []
    from_pe = 0
    
    for _, row in tqdm(onset_data.iterrows(), total=len(onset_data), desc='Building patient states'):
        qst = row[C_ONSET_TIME]  # flag for presumed infection
        icustayid = int(row[C_ICUSTAYID])
        assert qst > 0
        d1 = demog.loc[demog[C_ICUSTAYID] == icustayid, [C_AGE, C_DISCHTIME]].values.tolist()

        if d1[0][0] < 18:  # exclude younger than 18
            continue

        # Limit to a time period of -4h and +4h around the window of interest for sepsis3 definition)
        bounds = ((winb4 + 4) * 3600, (winaft + 4) * 3600)

        # Chart events, lab events, mech vent and extubated
        temp = time_window(chart_events.fetch(icustayid), C_CHARTTIME, qst, *bounds)
        temp2 = time_window(labU[labU[C_ICUSTAYID] == icustayid], C_CHARTTIME, qst, *bounds)
        temp3 = time_window(MV[MV[C_ICUSTAYID] == icustayid], C_CHARTTIME, qst, *bounds)
        if MV_procedure is not None:
            temp4 = time_window(MV_procedure[MV_procedure[C_ICUSTAYID] == icustayid], C_STARTTIME, qst, *bounds)
        else:
            temp4 = None
        
        # list of unique timestamps from all 3 sources / sorted in ascending order
        timesteps = sorted(pd.unique(pd.concat([
            temp[C_CHARTTIME], 
            temp2[C_CHARTTIME], 
            temp3[C_CHARTTIME]] + ([temp4[C_STARTTIME]] if temp4 is not None else []), ignore_index=True)))

        if len(timesteps) == 0:
            continue

        # Reformat each event, which may contain multiple items at the same timestep
        for i, timestep in enumerate(timesteps):

            # First three values: timestep, icustay ID, chart time
            item = {
                C_BLOC: i,
                C_ICUSTAYID: icustayid,
                C_TIMESTEP: timestep
            }
            
            # CHARTEVENTS: positions 4-31 (inclusive) are the 28 unique values
            # for different vitals, as stored in referenceMatrices['Refvitals']
            for _, event in temp[temp[C_CHARTTIME] == timestep].iterrows():
                if event[C_ITEMID] <= 0 or event[C_ITEMID] > len(CHART_FIELD_NAMES):
                    continue
               
                item[CHART_FIELD_NAMES[int(event[C_ITEMID] - 1)]] = event[C_VALUENUM]

            # LAB VALUES: positions 32-66 (inclusive) are the 35 unique values
            # for different lab tests, as stored in referenceMatrices['Reflabs']
            for _, event in temp2[temp2[C_CHARTTIME] == timestep].iterrows():
                if event[C_ITEMID] <= 0 or event[C_ITEMID] > len(LAB_FIELD_NAMES):
                    continue
                item[LAB_FIELD_NAMES[int(event[C_ITEMID] - 1)]] = event[C_VALUENUM]

            # MV: positions 67 and 68 store whether the measurement is relating
            # to being on a ventilator, and whether the patient was extubated
            matching_mv = (temp3[C_CHARTTIME] == timestep)
            if matching_mv.sum() > 0:
                if matching_mv.sum() > 1:
                    print(matching_mv.sum(), "MV items")
                event = temp3[matching_mv].iloc[0]
                item[C_MECHVENT] = event[C_MECHVENT]
                item[C_EXTUBATED] = event[C_EXTUBATED]
            
            # Second source of mechanical ventilation information
            if temp4 is not None and C_MECHVENT not in item and (temp4[C_STARTTIME] == timestep).sum() > 0:
                events = temp4[temp4[C_STARTTIME] == timestep]
                item[C_MECHVENT] = events[C_MECHVENT].any().astype(int)
                item[C_EXTUBATED] = events[C_EXTUBATED].any().astype(int)
                from_pe += 1
                
            combined_data.append(item)            

        infection_times.append({
            C_ICUSTAYID: icustayid,
            C_ONSET_TIME: qst,
            C_FIRST_TIMESTEP: timesteps[0], # first timestep
            C_LAST_TIMESTEP: timesteps[-1], # last timestep
            C_DISCHTIME: d1[0][1] # discharge time
        })
                
    print("Got {} items from procedure events".format(from_pe))
    state_df = pd.DataFrame(combined_data)
    qstime = pd.DataFrame(infection_times).set_index(C_ICUSTAYID)
    
    expected_columns = CHART_FIELD_NAMES + LAB_FIELD_NAMES
    for col in expected_columns:
        if col not in state_df.columns:
            print("Adding empty column '{}' (no data points)".format(col))
            state_df[col] = pd.NA
    return state_df, qstime

In [ ]:
PARENT_DIR = "./"
data_dir = os.path.join(PARENT_DIR)
out_dir = os.path.join(PARENT_DIR, 'intermediate_files')

In [ ]:
ce_paths = [os.path.join(data_dir, 'intermediate_files',path) for path in os.listdir(os.path.join(data_dir, 'intermediate_files')) if path.startswith("ce") and path.endswith(".csv")]
ce_dfs = [pd.read_csv(path) for path in ce_paths]

In [ ]:
chart_events = ChartEvents(ce_dfs)

In [ ]:
print("Reading onset data...")
onset_data = pd.read_csv(os.path.join(data_dir, 'intermediate_files', 'sepsis_onset.csv'))

In [ ]:
print("Reading demog...")
demog = pd.read_csv(os.path.join(data_dir, 'intermediate_files', 'demog.csv'))

In [ ]:
print("Reading labs...")
labU = pd.concat([pd.read_csv(os.path.join(data_dir, 'intermediate_files', 'labs_ce.csv')),
                  pd.read_csv(os.path.join(data_dir, 'intermediate_files', 'labs_le.csv'))
                 ], ignore_index=True)

In [ ]:
print("Reading mechvent...")
MV = pd.read_csv(os.path.join(data_dir, 'intermediate_files', 'mechvent.csv'))
MV_procedure = pd.read_csv(os.path.join(data_dir, 'processed_files', 'mechvent_pe.csv'))

In [ ]:
WINDOW_BEFORE = 25
WINDOW_AFTER = 49

In [ ]:
state_df, qstime = build_patient_states(chart_events,
                                        onset_data,
                                        demog,
                                        labU,
                                        MV,
                                        MV_procedure,
                                        WINDOW_BEFORE,
                                        WINDOW_AFTER
                                        )

In [ ]:
print("Result: state_df contains {} rows, {} columns".format(len(state_df), len(state_df.columns)))

In [ ]:
print("Data availability:")
for col in sorted(state_df.columns):
    nan_fraction = pd.isna(state_df[col]).sum() / len(state_df)
    if nan_fraction > 0.0:
        print(col, nan_fraction)
print("")

In [ ]:
state_df.to_csv(os.path.join(out_dir, "sepsis_cohort", 'patient_states.csv'), index=False, float_format='%g')
qstime.to_csv(os.path.join(out_dir, "sepsis_cohort", 'qstime.csv'), float_format='%g')

### 10. impute states

In [ ]:
df = pd.read_csv("./intermediate_files/sepsis_cohort/patient_states.csv")
old_df = df.copy()

In [ ]:
def remove_outliers(df):
    # Transfer temperatures tagged as celsius but obviously fahrenheit
    wrong_unit_temps = (df[C_TEMP_C] > 90) & pd.isna(df[C_TEMP_F])
    df.loc[wrong_unit_temps, C_TEMP_F] = df.loc[wrong_unit_temps, C_TEMP_C]

    # Multiply FiO2 to be in percentage instead of fraction
    df.loc[df[C_FIO2_100] < 1, C_FIO2_100] = df.loc[df[C_FIO2_100] < 1, C_FIO2_100] * 100

    df = fill_outliers(df, {
        # Vitals
        C_WEIGHT: (None, 300),      # weight
        C_HR: (None, 250),      # HR
        C_SYSBP: (0, 300),         # BP
        C_MEANBP: (0, 200),
        C_DIABP: (0, 200),
        C_RR: (None, 80),      # RR
        C_SPO2: (None, 150),     # SpO2
        C_TEMP_C: (None, 90),      # temp
        C_FIO2_100: (20, None),      # FiO2
        C_FIO2_1: (None, 1.5),
        C_O2FLOW: (None, 70),      # O2 flow
        C_PEEP: (0, 40),         # PEEP
        C_TIDALVOLUME: (None, 1800),    # Tidal volume
        C_MINUTEVENTIL: (None, 50),      # Minute volume
        
        # Labs
        C_POTASSIUM: (1, 15),         # K+
        C_SODIUM: (95, 178),       # Na+
        C_CHLORIDE: (70, 150),       # Cl
        C_GLUCOSE: (1, 1000),       # Glc
        C_CREATININE: (None, 150),     # creatinine
        C_MAGNESIUM: (None, 10),      # Mg
        C_CALCIUM: (None, 20),      # Ca
        C_IONISED_CA: (None, 5),       # ionized Ca
        C_CO2_MEQL: (None, 120),     # CO2
        C_SGPT: (None, 10000),   # SGPT
        C_SGOT: (None, 10000),   # SGOT
        C_HB: (None, 20),      # hemoglobin
        C_HT: (None, 65),      # hematocrit
        C_WBC_COUNT: (None, 500),     # WBC
        C_PLATELETS_COUNT: (None, 2000),    # platelet
        C_INR: (None, 20),      # INR
        C_ARTERIAL_PH: (6.7, 8),        # pH
        C_PAO2: (None, 700),     # PO2
        C_PACO2: (None, 200),     # PCO2
        C_ARTERIAL_BE: (-50, None),     # base excess
        C_ARTERIAL_LACTATE: (None, 30),      # lactate
    })

    # Clamp SpO2 to 100 after removing outliers
    df[df[C_SPO2] > 100] = 100
    
    return df

In [ ]:
print("Remove outliers")
df = remove_outliers(df)

In [ ]:
def estimate_gcs(rass):
    """
    Estimates the Glasgow Coma Scale value from the value of
    the Richmond Agitation Sedation Scale.
    """
    if rass >= 0: return 15
    elif rass == -1: return 14
    elif rass == -2: return 12
    elif rass == -3: return 11
    elif rass == -4: return 6
    elif rass == -5: return 3
    return pd.NA

In [ ]:
print("Estimate GCS from RASS")
ii = pd.isna(df[C_GCS])
df.loc[ii, C_GCS] = df.loc[ii, C_RASS].apply(estimate_gcs)

In [ ]:
def convert_fio2_units(df):
    """Converts FiO2 Set (a.k.a. FiO2_1) to FiOS (a.k.a. FiO2_100)."""
    # approximate that FiO2 Set (col 24) ~= FiO2 (col 23) / 100 
    missing_fio2_set = pd.isna(df[C_FIO2_1]) & ~pd.isna(df[C_FIO2_100])
    df.loc[missing_fio2_set, C_FIO2_1] = df.loc[missing_fio2_set, C_FIO2_100] / 100
    missing_fio2 = ~pd.isna(df[C_FIO2_1]) & pd.isna(df[C_FIO2_100])
    df.loc[missing_fio2, C_FIO2_100] = df.loc[missing_fio2, C_FIO2_1] * 100
    return df

In [ ]:
def estimate_fio2(df):
    # First fill in missing values
    df = convert_fio2_units(df)
    
    sah_fio2 = {}
    for col in [C_INTERFACE, C_FIO2_100, C_O2FLOW]:    
        print("SAH on " + col)
        sah_fio2[col] = sample_and_hold(df[C_ICUSTAYID], df[C_TIMESTEP], df[col], SAH_HOLD_DURATION[col])
        print("Eliminated {:.1f}% of NA values".format((1 - pd.isna(sah_fio2[col]).sum() / pd.isna(df[col]).sum()) * 100))
    
    # NO FiO2, YES O2 flow, no interface OR cannula
    print('NO FiO2, YES O2 flow, no interface OR cannula ', end='')
    mask = (pd.isna(sah_fio2[C_FIO2_100]) & ~pd.isna(sah_fio2[C_O2FLOW]) & ((sah_fio2[C_INTERFACE] == 0) | (sah_fio2[C_INTERFACE] == 2)))
    print('({} rows) '.format(mask.sum()), end='')
    df.loc[mask, C_FIO2_100] = fill_stepwise(sah_fio2[C_O2FLOW].loc[mask], zip(*(
        [15, 12, 10, 8, 6, 5, 4, 3, 2, 1],
        [70, 62, 55, 50, 44, 40, 36, 32, 28, 24]
    )))
    print('[DONE]')

    # NO FiO2, NO O2 flow, no interface OR cannula
    print('NO FiO2, NO O2 flow, no interface OR cannula ', end='')
    mask = (pd.isna(sah_fio2[C_FIO2_100]) & pd.isna(sah_fio2[C_O2FLOW]) & ((sah_fio2[C_INTERFACE] == 0) | (sah_fio2[C_INTERFACE] == 2)))
    print('({} rows) '.format(mask.sum()), end='')

    df.loc[mask, C_FIO2_100] = 21
    print('[DONE]')

    # NO FiO2, YES O2 flow, face mask OR.... OR ventilator (assume it's face mask)
    print('NO FiO2, YES O2 flow, face mask OR.... OR ventilator (assume it\'s face mask) ', end='')
    mask = (pd.isna(sah_fio2[C_FIO2_100]) & ~pd.isna(sah_fio2[C_O2FLOW]) & (pd.isna(sah_fio2[C_INTERFACE]) | sah_fio2[C_INTERFACE].isin((1, 3, 4, 5, 6, 9, 10))))
    print('({} rows) '.format(mask.sum()), end='')

    df.loc[mask, C_FIO2_100] = fill_stepwise(sah_fio2[C_O2FLOW].loc[mask], zip(*(
        [15, 12, 10, 8, 6, 4],
        [75, 69, 66, 58, 40, 36]
    )))
    print('[DONE]')

    # NO FiO2, NO O2 flow, face mask OR ....OR ventilator
    print('NO FiO2, NO O2 flow, face mask OR ....OR ventilator ', end='')
    mask = (pd.isna(sah_fio2[C_FIO2_100]) & pd.isna(sah_fio2[C_O2FLOW]) & 
            (pd.isna(sah_fio2[C_INTERFACE]) | sah_fio2[C_INTERFACE].isin((1, 3, 4, 5, 6, 9, 10))))
    print('({} rows) '.format(mask.sum()), end='')

    df.loc[mask, C_FIO2_100] = pd.NA
    print('[DONE]')

    # NO FiO2, YES O2 flow, Non rebreather mask
    print('NO FiO2, YES O2 flow, Non rebreather mask ', end='')
    mask = (pd.isna(sah_fio2[C_FIO2_100]) & ~pd.isna(sah_fio2[C_O2FLOW]) & sah_fio2[C_INTERFACE] == 7)
    print('({} rows) '.format(mask.sum()), end='')

    df.loc[mask, C_FIO2_100] = fill_stepwise(sah_fio2[C_O2FLOW].loc[mask], zip(*(
        [9.99, 8, 6],
        [80, 70, 60]
    )), zip(*(
        [10, 15],
        [90, 100]
    )))
    print('[DONE]')

    # NO FiO2, NO O2 flow, NRM
    print('NO FiO2, NO O2 flow, NRM ', end='')
    mask = (pd.isna(sah_fio2[C_FIO2_100]) & pd.isna(sah_fio2[C_O2FLOW]) & sah_fio2[C_INTERFACE] == 7)
    print('({} rows) '.format(mask.sum()), end='')

    df.loc[mask, C_FIO2_100] = pd.NA
    print('[DONE]')

    # update again FiO2 columns
    df = convert_fio2_units(df)
    print('[DONE]')
    return df

In [ ]:
print("Estimate FiO2")
df = estimate_fio2(df)

In [ ]:
def estimate_vitals(df):
    # BP - if we have two values, we can impute the others using the definition of mean BP
    print('BP ', end='')
    ii = ~pd.isna(df.loc[:, C_SYSBP]) & ~pd.isna(df.loc[:, C_MEANBP]) & pd.isna(df.loc[:, C_DIABP])

    df.loc[ii, C_DIABP] = (3 * df.loc[ii, C_MEANBP] - df.loc[ii, C_SYSBP]) / 2
    ii = ~pd.isna(df.loc[:, C_SYSBP]) & ~pd.isna(df.loc[:, C_DIABP]) & pd.isna(df.loc[:, C_MEANBP])
    df.loc[ii, C_MEANBP] = (df.loc[ii, C_SYSBP] + 2 * df.loc[ii, C_DIABP]) / 3
    ii = ~pd.isna(df.loc[:, C_MEANBP]) & ~pd.isna(df.loc[:, C_DIABP]) & pd.isna(df.loc[:, C_SYSBP])
    df.loc[ii, C_SYSBP] = 3 * df.loc[ii, C_MEANBP] - 2 * df.loc[ii, C_DIABP]
    print('[DONE]')

    # TEMP
    # some values recorded in the wrong column
    print('TEMP ', end='')
    ii = (df.loc[:, C_TEMP_F] > 25) & (df.loc[:, C_TEMP_F] < 45)  # tempF close to 37deg??!
    df.loc[ii, C_TEMP_C] = df.loc[ii, C_TEMP_F]
    df.loc[ii, C_TEMP_F] = np.nan
    ii = (df.loc[:, C_TEMP_C] > 70)  # tempC > 70?!!! probably degF
    df.loc[ii, C_TEMP_F] = df.loc[ii, C_TEMP_C]
    df.loc[ii, C_TEMP_C] = np.nan
    
    ii = ~pd.isna(df.loc[:, C_TEMP_C]) & pd.isna(df.loc[:, C_TEMP_F])
    df.loc[ii, C_TEMP_F] = df.loc[ii, C_TEMP_C] * 1.8 + 32
    ii = ~pd.isna(df.loc[:, C_TEMP_F]) & pd.isna(df.loc[:, C_TEMP_C])
    df.loc[ii, C_TEMP_C] = (df.loc[ii, C_TEMP_F] - 32) / 1.8
    print('[DONE]')

    # Hb/Ht
    print('Hb/Ht ', end='')
    ii = ~pd.isna(df.loc[:, C_HB]) & pd.isna(df.loc[:, C_HT])
    df.loc[ii, C_HT] = (df.loc[ii, C_HB] * 2.862) + 1.216
    ii = ~pd.isna(df.loc[:, C_HT]) & pd.isna(df.loc[:, C_HB])
    df.loc[ii, C_HB] = (df.loc[ii, C_HT] - 1.216) / 2.862
    print('[DONE]')

    # BILI
    print('BILI ', end='')
    ii = ~pd.isna(df.loc[:, C_TOTAL_BILI]) & pd.isna(df.loc[:, C_DIRECT_BILI])
    df.loc[ii, C_DIRECT_BILI] = (df.loc[ii, C_TOTAL_BILI] * 0.6934) - 0.1752
    ii = ~pd.isna(df.loc[:, C_DIRECT_BILI]) & pd.isna(df.loc[:, C_TOTAL_BILI])
    df.loc[ii, C_TOTAL_BILI] = (df.loc[ii, C_DIRECT_BILI] + 0.1752) / 0.6934
    print('[DONE]')
    
    return df

In [ ]:
print("Estimate vitals")
df = estimate_vitals(df)

In [ ]:
print("Sample and hold")
sah_series = {C_BLOC: df[C_BLOC],
              C_ICUSTAYID: df[C_ICUSTAYID],
              C_TIMESTEP: df[C_TIMESTEP]
             }
for col in SAH_FIELD_NAMES:
    print("SAH on " + col)
    sah_series[col] = sample_and_hold(df[C_ICUSTAYID],
                                      df[C_TIMESTEP],
                                      df[col],
                                      SAH_HOLD_DURATION[col],
                                      col_name=col)
    print("Eliminated {:.1f}% of NA values".format((1 - pd.isna(sah_series[col]).sum() / pd.isna(df[col]).sum()) * 100))

df = pd.DataFrame(sah_series)
del sah_series

In [ ]:
print("Write")
df.to_csv("./intermediate_files/sepsis_cohort/patient_states_filled.csv", index=False, float_format='%g')

#### 이거 오류나서 일단 패스 (안필요할 것 같아서)

In [ ]:
# print("Write mask file")
        
# # Compare the old dataframe to the new one and see where values have cropped up
# mask_series = {
#     C_BLOC: df[C_BLOC],
#     C_ICUSTAYID: df[C_ICUSTAYID],
#     C_TIMESTEP: df[C_TIMESTEP]
# }

# for col in set(old_df.columns) & set(df.columns):
#     if col in mask_series: continue
#     old = old_df[col]
#     new = df[col]
#     mask_series[col] = np.where((pd.isna(old) & ~pd.isna(new)) | (pd.notna(old) & pd.notna(new) & (old != new)), 1, np.where(~pd.isna(old) & pd.isna(new), -1, 0))
    
# pd.DataFrame(mask_series).to_csv("./intermediate_files/sepsis_cohort/state_imputation_mask.csv", index=False)

### 11. build states and actions

In [ ]:
print("Reading states...")
df = pd.read_csv("./intermediate_files/sepsis_cohort/patient_states_filled.csv")
qstime = pd.read_csv("./intermediate_files/sepsis_cohort/qstime.csv")
qstime = qstime.set_index(C_ICUSTAYID, drop=True)

In [ ]:
print("Reading data files...")
demog = pd.read_csv("./intermediate_files/demog.csv")
inputpreadm = pd.read_csv("./processed_files/preadm_fluid.csv")
inputMV = pd.read_csv("./intermediate_files/fluid_align_with_snuh_mv.csv")
vasoMV = pd.read_csv("./intermediate_files/vaso_align_with_snuh_mv.csv")
note = pd.read_csv("./intermediate_files/note.csv")
UOpreadm = pd.read_csv("./processed_files/preadm_uo.csv")
UO = pd.read_csv("./processed_files/uo.csv")

In [ ]:
def build_states_and_actions(df, qstime, inputMV, inputCV, inputpreadm, vasoMV, vasoCV, demog, note, UOpreadm, UO, timestep_resolution, winb4, winaft, head=None, allowed_stays=None):
    """
    Performs two tasks: bins the data into time intervals defined by 
    timestep_resolution, and adds input and output information (vasopressors,
    fluids, and urine output).
    
    inputCV and vasoCV may be None if using MIMIC-IV (contains MetaVision only).
    """
    icustayidlist = np.unique(df[C_ICUSTAYID])
    icustayidlist = sorted(icustayidlist[~pd.isna(icustayidlist)])
    if allowed_stays is not None:
        old_count = len(icustayidlist)
        allowed_stays = set(allowed_stays)
        icustayidlist = [id for id in icustayidlist if id in allowed_stays]
        print("Filtered from {} to {} ICU stay ids".format(old_count, len(icustayidlist)))
    else:
        print("{} ICU stay IDs".format(len(icustayidlist)))
    if head:
        icustayidlist = icustayidlist[:head]

    combined_data = []
    bin_indexes = pd.Series(index=np.arange(len(df)), dtype=pd.Int64Dtype()) # Pointers corresponding to each row in df that point to the index of combined_data that this row went into
    
    # -52 until +28 = 80 hours in total
    total_duration = (winb4 + 3) + (winaft + 3)
    for icustayid in tqdm(icustayidlist, desc='Building states and actions'):

        # CHARTEVENTS AND LAB VALUES
        temp = df.loc[df[C_ICUSTAYID] == icustayid, :]  # subtable of interest
        beg = temp[C_TIMESTEP].iloc[0]  # timestamp of first record

        # IV FLUID STUFF
        input = inputMV.loc[inputMV[C_ICUSTAYID] == icustayid, :]  # subset of interest
        if inputCV is not None:
            input2 = inputCV.loc[inputCV[C_ICUSTAYID] == icustayid, :]  # subset of interest
        else:
            input2 = None
        startt = input[C_STARTTIME]  # start of all infusions and boluses
        endt = input[C_ENDTIME]  # end of all infusions and boluses
        rate = input[C_NORM_INFUSION_RATE]  # normalized rate of infusion (is NaN for boluses) || corrected for tonicity

        pread = inputpreadm.loc[inputpreadm[C_ICUSTAYID] == icustayid, C_INPUT_PREADM]  # preadmission volume
        if not pread.empty:  # store the value, if available
            totvol = pread.sum()
        else:
            totvol = 0  # if not documented: it's zero

        # compute volume of fluid given before start of record!!!
        t0 = 0
        t1 = beg
        # input from MV (4 ways to compute)
        infu = np.nansum(rate * (endt - startt) * ((endt <= t1) & (startt >= t0)) / 3600 +
                        rate * (endt - t0) * ((startt <= t0) & (endt <= t1) & (endt >= t0)) / 3600 +
                        rate * (t1 - startt) * ((startt >= t0) & (endt >= t1) & (startt <= t1)) / 3600 +
                        rate * (t1 - t0) * ((endt >= t1) & (startt <= t0)) / 3600)
        # all boluses received during this timestep, from inputMV (rate is always NaN for boluses)
        bolusMV = np.nansum(input.loc[pd.isna(input[C_RATE]) & (input[C_STARTTIME] >= t0) & (input[C_STARTTIME] <= t1), C_TEV])
        if input2 is not None:
            bolusCV = np.nansum(input2.loc[(input2[C_CHARTTIME] >= t0) & (input2[C_CHARTTIME] <= t1), C_TEV])
            bolus = bolusMV + bolusCV
        else:
            bolus = bolusMV
        totvol = np.nansum([totvol, infu, bolus])

        # VASOPRESSORS
        vaso1 = vasoMV.loc[vasoMV[C_ICUSTAYID] == icustayid, :]  # subset of interest
        if vasoCV is not None:
            vaso2 = vasoCV.loc[vasoCV[C_ICUSTAYID] == icustayid, :]  # subset of interest
        else:
            vaso2 = None
        startv = vaso1[C_STARTTIME]  # start of VP infusion
        endv = vaso1[C_ENDTIME]  # end of VP infusions
        ratev = vaso1[C_RATESTD]  # rate of VP infusion

        # note
        note1 = note.loc[note[C_ICUSTAYID] == icustayid, :]  # subset of interest
        
        # DEMOGRAPHICS / gender, age, elixhauser, re-admit, died in hosp?, died within
        # 48h of out_time (likely in ICU or soon after), died within 90d after admission?
        demog_row = demog[demog[C_ICUSTAYID] == icustayid].iloc[0]
        dem = {
            C_GENDER: demog_row[C_GENDER],
            C_AGE: demog_row[C_AGE],
            C_ELIXHAUSER: demog_row[C_ELIXHAUSER],
            C_RE_ADMISSION: demog_row[C_ADM_ORDER] > 1,
            C_DIED_IN_HOSP: demog_row[C_MORTA_HOSP],
            C_DIED_WITHIN_48H_OF_OUT_TIME: abs((demog_row[C_DOD]) - (demog_row[C_OUTTIME])) < (24 * 3600 * 2),
            C_MORTA_90: demog_row[C_MORTA_90],
            C_DELAY_END_OF_RECORD_AND_DISCHARGE_OR_DEATH: (qstime.loc[icustayid, C_DISCHTIME] - qstime.loc[icustayid, C_LAST_TIMESTEP]) / 3600
        }
        
        # URINE OUTPUT
        output = UO.loc[UO[C_ICUSTAYID] == icustayid, :]  # urine output for this icu stay
        pread = UOpreadm.loc[UOpreadm[C_ICUSTAYID] == icustayid, C_VALUE]  # preadmission UO
        if not pread.empty:  # store the value, if available
            UOtot = pread.sum()
        else:
            UOtot = 0
        # adding the volume of urine produced before start of recording! this could be
        # during the ICU stay but before the sepsis window
        UOnow = np.sum(output.loc[(output[C_CHARTTIME] >= t0) & (output[C_CHARTTIME] <= t1), C_VALUE])  # t0 and t1 defined above
        UOtot += UOnow

        for j in np.arange(0, total_duration, timestep_resolution):
            t0 = 3600 * j + beg  # left limit of time window
            t1 = 3600 * (j + timestep_resolution) + beg  # right limit of time window
            value = temp.loc[(temp[C_TIMESTEP] >= t0) & (temp[C_TIMESTEP] <= t1), :]  # index of items in this time period
            if len(value) == 0:
                continue
                
            # ICUSTAY_ID, OUTCOMES, DEMOGRAPHICS
            item = {
                C_BLOC: (j / timestep_resolution) + 1,  # 'bloc' = timestep (1,2,3...)
                C_ICUSTAYID: icustayid,       # icustay_ID
                C_TIMESTEP: int(3600 * j + beg),  # t0 = lower limit of time window
            }
            # Columns 4-11: demographics and outcomes
            item.update(dem)

            # #####################   DISCUSS ADDING STUFF HERE / RANGE, MIN, MAX ETC   ################

            # Columns 12-76: mean of chart and lab values in window
            # Are there categorical items in here?
            item.update({col: value[col].mean(skipna=True) for col in SAH_FIELD_NAMES})
            # shock index = HR/SBP and P/F (not yet computed)

            # VASOPRESSORS
            # for CV: dose at timestamps.
            # for MV: 4 possibles cases, each one needing a different way to compute the dose of VP actually administered:
            # ----t0---start----end-----t1----
            # ----start---t0----end----t1----
            # -----t0---start---t1---end
            # ----start---t0----t1---end----

            # MV
            v = (((endv >= t0) & (endv <= t1)) | ((startv >= t0) & (endv <= t1)) | ((startv >= t0) & (startv <= t1)) | ((startv <= t0) & (endv >= t1)))
            if vaso2 is not None:
                v2 = vaso2.loc[(vaso2[C_CHARTTIME] >= t0) & (vaso2[C_CHARTTIME] <= t1), C_RATESTD]
            else:
                v2 = pd.Series([], dtype=np.float64)

            if not ratev.loc[v].empty or not v2.empty:
                all_vs = np.concatenate([ratev.loc[v].values, v2.values])
                all_vs = all_vs[~np.isnan(all_vs)]
                if all_vs.size > 0:
                    item[C_MEDIAN_DOSE_VASO] = np.nanmedian(all_vs)
                    item[C_MAX_DOSE_VASO] = np.nanmax(all_vs)

            # INPUT FLUID
            # input from MV (4 ways to compute)
            infu = np.nansum(rate * (endt - startt) * ((endt <= t1) & (startt >= t0)) / 3600 +
                            rate * (endt - t0) * ((startt <= t0) & (endt <= t1) & (endt >= t0)) / 3600 +
                            rate * (t1 - startt) * ((startt >= t0) & (endt >= t1) & (startt <= t1)) / 3600 +
                            rate * (t1 - t0) * ((endt >= t1) & (startt <= t0)) / 3600)
            # all boluses received during this timestep, from inputMV (need to check rate is NaN) and inputCV (simpler):
            bolusMV = np.nansum(input.loc[pd.isna(input[C_RATE]) & (input[C_STARTTIME] >= t0) & (input[C_STARTTIME] <= t1), C_TEV])
            if input2 is not None:
                bolusCV = np.nansum(input2.loc[(input2[C_CHARTTIME] >= t0) & (input2[C_CHARTTIME] <= t1), C_TEV])
                bolus = bolusMV + bolusCV
            else:
                bolus = bolusMV
            
            # sum fluid given
            totvol = np.nansum([totvol, infu, bolus])
            item[C_INPUT_TOTAL] = totvol # total fluid
            item[C_INPUT_STEP] = infu + bolus # fluid at this step

            # UO
            UOnow = np.nansum(output.loc[(output[C_CHARTTIME] >= t0) & (output[C_CHARTTIME] <= t1), C_VALUE])
            UOtot = np.nansum([UOtot, UOnow])
            item[C_OUTPUT_TOTAL] = UOtot  # total UO
            item[C_OUTPUT_STEP] = UOnow  # UO at this step

            # CUMULATED BALANCE
            item[C_CUMULATED_BALANCE] = totvol - UOtot  # cumulated balance

            # Clinical note
            text_note = note1.loc[(note1[C_CHARTTIME] >= t0) & (note1[C_CHARTTIME] <= t1), "text"].values
            if len(text_note) >0:
                text_note = "".join(text_note)
                item[C_TEXT] = text_note
            else:
                text_note = "None"
                item[C_TEXT] = text_note
            
            bin_indexes[value.index] = len(combined_data) # Point to this row
            combined_data.append(item)

    result = pd.DataFrame(combined_data)
    expected_columns = DEMOGRAPHICS_FIELD_NAMES + SAH_FIELD_NAMES + IO_FIELD_NAMES + COMPUTED_FIELD_NAMES
    for col in expected_columns:
        if col not in result.columns:
            print("Adding empty column '{}' (no data points)".format(col))
            result[col] = pd.NA

    mapping_df = pd.DataFrame({
        C_BLOC: df[C_BLOC],
        C_ICUSTAYID: df[C_ICUSTAYID],
        C_TIMESTEP: df[C_TIMESTEP],
        C_BIN_INDEX: bin_indexes
    })
    return result, mapping_df

In [ ]:
inputCV = None
vasoCV = None
RESOLUTION = 4.0 #per time sampling
WINDOW_BEFORE = 25
WINDOW_AFTER = 49

In [ ]:
result, mapping = build_states_and_actions(df,
                                           qstime,
                                           inputMV,
                                           inputCV,
                                           inputpreadm,
                                           vasoMV,
                                           vasoCV,
                                           demog,
                                           note,
                                           UOpreadm,
                                           UO,
                                           RESOLUTION,
                                           WINDOW_BEFORE,
                                           WINDOW_AFTER,
                                           head=None,
                                           allowed_stays=None
                                            )

In [ ]:
print("Writing to file")
result.to_csv("./intermediate_files/sepsis_cohort/states_and_actions.csv", index=False, float_format='%g')
mapping.to_csv("./intermediate_files/sepsis_cohort/bin_mapping.csv", index=False, float_format='%g')

### 12. impute states actions

In [ ]:
df = pd.read_csv("./intermediate_files/sepsis_cohort/states_and_actions.csv")
old_df = df.copy()

In [ ]:
def correct_features(df):
    # CORRECT GENDER
    df[C_GENDER] = df[C_GENDER] - 1

    # CORRECT AGE > 200 yo
    ii = df[C_AGE] > 150
    df.loc[ii, C_AGE] = 91

    # FIX MECHVENT
    df.loc[pd.isna(df[C_MECHVENT]), C_MECHVENT] = 0
    df.loc[df[C_MECHVENT] > 0, C_MECHVENT] = 1

    # FIX Elixhauser missing values
    df.loc[pd.isna(df[C_ELIXHAUSER]), C_ELIXHAUSER] = np.nanmedian(df[C_ELIXHAUSER])  # use the median value / only a few missing data points

    # vasopressors / no NAN
    df.loc[pd.isna(df[C_MEDIAN_DOSE_VASO]), C_MEDIAN_DOSE_VASO] = 0
    df.loc[pd.isna(df[C_MAX_DOSE_VASO]), C_MAX_DOSE_VASO] = 0
    
    return df    

In [ ]:
print("Correcting features")
df = correct_features(df)

In [ ]:
print("Linear interpolation")
miss = pd.isna(df).sum() / len(df)
impute_columns = (miss > 0) & (miss < 0.05)  # less than 5# missingness

In [ ]:
for col in df.loc[:,impute_columns].columns:
    print("Linear interpolation on", col)
    new_vals = fixgaps(df[col])
    df[col] = new_vals

In [ ]:
print("knn imputation")
knn_cols = CHART_FIELD_NAMES + LAB_FIELD_NAMES
df[knn_cols] = knn_impute(df[knn_cols], na_threshold=0.9, provenance=None)

In [ ]:
print("Computing P/F")
df[C_PAO2_FIO2] = compute_pao2_fio2(df)
print("Computing shock index")
df[C_SHOCK_INDEX] = compute_shock_index(df)
print("Computing SOFA")
df[C_SOFA] = compute_sofa(df, timestep_resolution=RESOLUTION)
print("Computing SIRS")
df[C_SIRS] = compute_sirs(df)

In [ ]:
print("Write")
df.to_csv("./intermediate_files/sepsis_cohort/states_and_actions_filled.csv", index=False, float_format='%g')

#### 오류나서 일단 패스 (나중에 안필요할 것 같음)

In [ ]:
# print("Write mask file")
        
# # Compare the old dataframe to the new one and see where values have cropped up
# mask_series = {
#     C_BLOC: df[C_BLOC],
#     C_ICUSTAYID: df[C_ICUSTAYID],
#     C_TIMESTEP: df[C_TIMESTEP]
# }

# for col in set(old_df.columns) & set(df.columns):
#     if col in mask_series: continue
#     old = old_df[col]
#     new = df[col]
#     mask_series[col] = np.where((pd.isna(old) & ~pd.isna(new)) | (~pd.isna(old) & (old != new)), 1, np.where(~pd.isna(old) & pd.isna(new), -1, 0))
    
# pd.DataFrame(mask_series).to_csv("./intermediate_files/sepsis_cohort/states_and_actions_mask.csv", index=False)

### 13. build sepsis cohort

In [ ]:
out_dir = "./preprocessed_files"

In [ ]:
df = pd.read_csv("./intermediate_files/sepsis_cohort/states_and_actions_filled.csv")
qstime = pd.read_csv("./intermediate_files/sepsis_cohort/qstime.csv")
qstime = qstime.set_index(C_ICUSTAYID, drop=True)

In [ ]:
print("Before filtering:", len(set(df[C_ICUSTAYID])), "ICU stays")  # count before

### albumin, arterial_be, PT, PTT, BUN, HCO3 내가 정보 찾아보고 말이 안되는거 같은거 처리시킴. 임상적으로 올바른지 확인 필요

In [ ]:
def outlier_stay_ids(df):
    """Returns a list of ICU stay IDs that should be removed from the dataset."""
    outliers = set()

    # some have Albumin > 10
    outliers |= set(df[df[C_ALBUMIN] > 10][C_ICUSTAYID].unique())
    
    # some have arterial be > 50
    outliers |= set(df[df[C_ARTERIAL_BE] > 50][C_ICUSTAYID].unique())
    
    # some have PT > 200 
    outliers |= set(df[df[C_PT] > 200][C_ICUSTAYID].unique())
    
    # some have PTT > 200 -> ex.250112
    outliers |= set(df[df[C_PTT] > 200][C_ICUSTAYID].unique())
    
    # some have BUN > 150
    outliers |= set(df[df[C_BUN] > 150][C_ICUSTAYID].unique())
    
    # some have HCO3 > 100
    outliers |= set(df[df[C_HCO3] > 100][C_ICUSTAYID].unique())
    
    # check for patients with extreme UO = outliers = to be deleted (>40 litres of UO per 4h!!)
    outliers |= set(df[df[C_OUTPUT_STEP] > 12000][C_ICUSTAYID].unique())

    # some have bili = 999999
    outliers |= set(df[df[C_TOTAL_BILI] > 10000][C_ICUSTAYID].unique())

    # check for patients with extreme INTAKE = outliers = to be deleted (>10 litres of intake per 4h!!)
    outliers |= set(df[df[C_INPUT_STEP] > 10000][C_ICUSTAYID].unique())

    return outliers

In [ ]:
outliers = outlier_stay_ids(df)
print(len(outliers), "outliers to remove")
df = df[~df[C_ICUSTAYID].isin(outliers)]

In [ ]:
def treatment_stopped_stay_ids(df):
    a = df[[C_BLOC, C_ICUSTAYID, C_MORTA_90, C_MAX_DOSE_VASO, C_SOFA]]
    grouped = a.groupby(C_ICUSTAYID)
    d = pd.merge(grouped.agg('max'),
                grouped.size().rename(C_NUM_BLOCS),
                how='left',
                left_index=True,
                right_index=True).drop(C_BLOC, axis=1)
    last_bloc = a.sort_values(C_BLOC, ascending=False).drop_duplicates(C_ICUSTAYID).rename({
        C_MAX_DOSE_VASO: C_LAST_VASO,
        C_SOFA: C_LAST_SOFA
    }, axis=1).drop(C_MORTA_90, axis=1)
    d = pd.merge(d,
                last_bloc,
                how='left',
                left_index=True,
                right_on=C_ICUSTAYID).set_index(C_ICUSTAYID, drop=True)
    
    stopped_treatment = d[
        (d[C_MORTA_90] == 1) & 
        (pd.isna(d[C_LAST_VASO]) | (d[C_LAST_VASO] < 0.01)) &
        (d[C_MAX_DOSE_VASO] > 0.3) &
        (d[C_LAST_SOFA] >= d[C_SOFA] / 2) &
        (d[C_NUM_BLOCS] < 20)
    ].index
    
    return stopped_treatment

In [ ]:
stopped_treatment = treatment_stopped_stay_ids(df)
print(len(stopped_treatment), "stays to remove because treatment was stopped and patient died")
df = df[~df[C_ICUSTAYID].isin(stopped_treatment)]

In [ ]:
def died_in_icu_stay_ids(df):
    # exclude patients who died in ICU during data collection period
    died_in_icu = df[
        (df[C_DIED_WITHIN_48H_OF_OUT_TIME] == 1) & (df[C_DELAY_END_OF_RECORD_AND_DISCHARGE_OR_DEATH] < 24)
    ][C_ICUSTAYID].unique()
    return died_in_icu

In [ ]:
died_in_icu = died_in_icu_stay_ids(df)    
print(len(died_in_icu), "patients to remove because died in ICU during data collection")
df = df[~df[C_ICUSTAYID].isin(died_in_icu)]

In [ ]:
print("After filtering:", len(set(df[C_ICUSTAYID])), "ICU stays")  # count after

In [ ]:
sepsis = df.groupby('icustayid').agg({C_MORTA_90: 'first',
                                      C_SOFA: 'max',
                                      C_SIRS: 'max',
                                     }).rename({C_SOFA: C_MAX_SOFA, 
                                                C_SIRS: C_MAX_SIRS,
                                               }, axis=1)
sepsis = sepsis[sepsis[C_MAX_SOFA] >= 2]
print(len(sepsis), "patients with max SOFA >= 2")

In [ ]:
sepsis = pd.merge(sepsis, qstime[C_ONSET_TIME], how='left', left_index=True, right_index=True)

In [ ]:
print("Write")
sepsis.to_csv(os.path.join(out_dir, "sepsis_cohort.csv"))

In [ ]:
from collections import Counter
Counter(sepsis['morta_90'])

In [ ]:
sepsis